# CodeGen — Group 45
## Step 2: The Data Engine — building validated Python→Rust pairs

**What we are building:** the training data for our project. We take simple Python problems,
translate each into Rust with a small coder model, and **keep only the Rust that compiles and
passes the tests** — using the exact harness from Step 1 as the filter. The output is
`pairs.jsonl`, a file of verified (Python → Rust) examples we will fine-tune on in Step 3.

**The key idea:** we don't *find* a Python→Rust corpus (one barely exists). We **generate and
validate** it — the MultiPL-T recipe. A noisy translator is fine, because the harness throws
away everything that doesn't pass.

**Leakage rule:** we build training data from **MBPP** problems and will **evaluate on
HumanEval-Rust** (Step 1) — two disjoint problem sets, so we never train on what we test.

**To run:** needs a **GPU** (`Runtime → Change runtime type → T4 GPU`). Then `Runtime → Run all`.


## 1. Install Rust + Python libraries
Note: we do **not** upgrade `torch` — Colab ships a matched `torch`/`torchvision` pair, and
force-upgrading torch breaks it (the `torchvision::nms` error). We only add transformers etc.

In [1]:
!curl https://sh.rustup.rs -sSf | sh -s -- -y -q
import os
os.environ["PATH"] = os.path.expanduser("~/.cargo/bin") + ":" + os.environ["PATH"]
!rustc --version
# IMPORTANT: do NOT add `torch` here — upgrading it breaks Colab's torchvision.
!pip install -q -U datasets transformers accelerate
print("setup done")


warn: It looks like you have an existing rustup settings file at:
warn: /root/.rustup/settings.toml
warn: Rustup will install the default toolchain as specified in the settings file,
warn: instead of the one inferred from the default host triple.

  stable-x86_64-unknown-linux-gnu installed - rustc 1.96.1 (31fca3adb 2026-06-26)


Rust is installed now. Great!

To get started you may need to restart your current shell.
This would reload your PATH environment variable to include
Cargo's bin directory ($HOME/.cargo/bin).

To configure your current shell, you need to source
the corresponding env file under $HOME/.cargo.

This is usually done by running one of the following (note the leading DOT):
. "$HOME/.cargo/env"            # For sh/bash/zsh/ash/dash/pdksh
source "$HOME/.cargo/env.fish"  # For fish
source "~/.cargo/env.nu"  # For nushell
source "$HOME/.cargo/env.tcsh"  # For tcsh
. "$HOME/.cargo/env.ps1"        # For pwsh
source "$HOME/.cargo/env.xsh"   # For xonsh
rustc 1.96.1 (31fca3

## 2. The harness (same as Step 1 — our validator)
This is the unchanged `evaluate_one` from Step 1. Here it plays a new role: the **quality
filter** that decides which generated Rust is good enough to keep as training data.

In [2]:
import subprocess, tempfile, os

def evaluate_one(prompt, completion, tests, compile_timeout=60, run_timeout=10):
    program = prompt + completion + tests
    with tempfile.TemporaryDirectory() as wd:
        src  = os.path.join(wd, "main.rs")
        binp = os.path.join(wd, "prog")
        with open(src, "w") as f:
            f.write(program)
        try:
            c = subprocess.run(["rustc", src, "-o", binp],
                               capture_output=True, text=True, timeout=compile_timeout)
        except subprocess.TimeoutExpired:
            return "compile_timeout"
        if c.returncode != 0:
            return "compile_error"
        try:
            r = subprocess.run([binp], capture_output=True, text=True, timeout=run_timeout)
        except subprocess.TimeoutExpired:
            return "run_timeout"
        return "pass" if r.returncode == 0 else "run_fail"

print("harness ready")


harness ready


## 3. Load training problems (MBPP-Rust) + their Python solutions
- **mbpp-rs** (from MultiPL-E) gives us 354 Rust problems, each with a signature and **ready Rust
  unit tests** — perfect for validation.
- **MBPP** (the original Python dataset) gives us the **Python solution** for each problem.

We join them by task id (the `mbpp_<id>_...` in the Rust name matches MBPP's `task_id`).

In [3]:
from datasets import load_dataset
import re

def load_rs(cfg):
    try:
        return load_dataset("nuprl/MultiPL-E", cfg, split="test")
    except Exception:
        return load_dataset("nuprl/MultiPL-E", cfg, split="test", trust_remote_code=True)

train_rs = load_rs("mbpp-rs")          # Rust problems + tests (our training source)

# Python solutions from MBPP, across all splits, keyed by task_id
mbpp = load_dataset("google-research-datasets/mbpp", "full")
py_by_id = {}
for split in mbpp:
    for ex in mbpp[split]:
        py_by_id[ex["task_id"]] = ex["code"]

def mbpp_id(name):
    m = re.match(r"mbpp_(\d+)_", name)
    return int(m.group(1)) if m else None

problems = []
for ex in train_rs:
    pid = mbpp_id(ex["name"])
    problems.append({"name": ex["name"], "prompt": ex["prompt"], "tests": ex["tests"],
                     "stop_tokens": ex["stop_tokens"], "task_id": pid,
                     "python": py_by_id.get(pid)})

have_py = sum(p["python"] is not None for p in problems)
print(len(problems), "Rust training problems;", have_py, "matched to a Python solution")
print("\n=== Example problem ===")
print("PYTHON:\n", problems[0]["python"])
print("RUST PROMPT:\n", problems[0]["prompt"])
print("RUST TESTS:\n", problems[0]["tests"][:200], "...")


README.md:   0%|          | 0.00/33.2k [00:00<?, ?B/s]

mbpp-rs/test-00000-of-00001.parquet:   0%|          | 0.00/72.1k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/354 [00:00<?, ? examples/s]

README.md:   0%|          | 0.00/9.06k [00:00<?, ?B/s]

full/train-00000-of-00001.parquet:   0%|          | 0.00/87.2k [00:00<?, ?B/s]

...

Generating train split:   0%|          | 0/374 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/500 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/90 [00:00<?, ? examples/s]

Generating prompt split:   0%|          | 0/10 [00:00<?, ? examples/s]

354 Rust training problems; 354 matched to a Python solution

=== Example problem ===
PYTHON:
 import math
def is_not_prime(n):
    result = False
    for i in range(2,int(math.sqrt(n)) + 1):
        if n % i == 0:
            result = True
    return result
RUST PROMPT:
 /// Write a rsthon function to identify non-prime numbers.
fn is_not_prime(n: isize) -> bool {

RUST TESTS:
 }

fn main() {
    let candidate = is_not_prime;
    assert_eq!(candidate(2), false);
    assert_eq!(candidate(10), true);
    assert_eq!(candidate(35), true);
    assert_eq!(candidate(37), false);
}
 ...


## 4. Load the translator model
We use a small but capable coder model — **Qwen2.5-Coder-1.5B** — to do the Python→Rust
translation. It runs on a free T4. (You can swap `TRANSLATOR` for a bigger model or an API
later to raise the yield — the harness filter stays the same either way.)

In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

TRANSLATOR = "Qwen/Qwen2.5-Coder-1.5B"
dtype = torch.float16 if torch.cuda.is_available() else torch.float32
ttok = AutoTokenizer.from_pretrained(TRANSLATOR)
tmodel = AutoModelForCausalLM.from_pretrained(TRANSLATOR, torch_dtype=dtype)
tmodel = tmodel.to("cuda" if torch.cuda.is_available() else "cpu")
print("translator loaded on", tmodel.device)


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.31k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/117 [00:00<?, ?B/s]

translator loaded on cuda:0


## 5. The engine: translate → validate → keep
For each problem we build a prompt that shows the model the **Python as a reference comment**
and then the **Rust signature to complete** (this "parallel pairing" is what the scaling-laws
paper recommends). The model writes the Rust body; we trim it at the stop token; we run it
through the harness; and we **keep only the ones that pass**.

In [5]:
import json, os

def python_as_comment(py):
    if not py:
        return ""
    body = "\n".join("// " + line for line in py.strip().splitlines())
    return "// Reference Python implementation:\n" + body + "\n"

def trim_to_body(text):
    # Cut at the brace that closes the function, IGNORING braces inside strings/chars/comments.
    depth = 1
    i, n = 0, len(text)
    in_str = in_char = in_line = in_block = False
    while i < n:
        ch = text[i]
        nxt = text[i+1] if i+1 < n else ""
        if in_line:
            if ch == "\n": in_line = False
            i += 1; continue
        if in_block:
            if ch == "*" and nxt == "/": in_block = False; i += 2; continue
            i += 1; continue
        if in_str:
            if ch == "\\": i += 2; continue
            if ch == '"': in_str = False
            i += 1; continue
        if in_char:
            if ch == "\\": i += 2; continue
            if ch == "'": in_char = False
            i += 1; continue
        if ch == "/" and nxt == "/": in_line = True; i += 2; continue
        if ch == "/" and nxt == "*": in_block = True; i += 2; continue
        if ch == '"': in_str = True; i += 1; continue
        if ch == "'":
            if nxt == "\\" or (i+2 < n and text[i+2] == "'"): in_char = True
            i += 1; continue
        if ch == "{": depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0: return text[:i]
        i += 1
    return text

def translate_candidates(problem, n=8, max_new_tokens=512, temperature=0.8, batch=4):
    """Generate n diverse Rust attempts for one problem (sampling). Batched to avoid OOM."""
    prompt = python_as_comment(problem["python"]) + problem["prompt"]
    inputs = ttok(prompt, return_tensors="pt").to(tmodel.device)
    cands, remaining = [], n
    while remaining > 0:
        k = min(batch, remaining)
        out = tmodel.generate(**inputs, max_new_tokens=max_new_tokens,
                              do_sample=True, temperature=temperature, top_p=0.95,
                              num_return_sequences=k, pad_token_id=ttok.eos_token_id)
        for seq in out:
            text = ttok.decode(seq[inputs.input_ids.shape[1]:], skip_special_tokens=True)
            cands.append(trim_to_body(text))
        remaining -= k
    return cands

def build_pairs(limit=None, n=8, out_path="pairs_v2.jsonl"):
    """MultiPL-T style: many samples per problem, keep EVERY one that passes.

    Crash-safe: each kept pair is appended to out_path and flushed immediately, and any
    problem already present in out_path is skipped. So if Colab disconnects mid-run,
    just re-run this cell and it resumes where it stopped. Writes to a NEW file
    (pairs_v2.jsonl) so the original 194-pair pairs.jsonl is never overwritten.
    """
    todo = problems if limit is None else problems[:limit]

    done = set()
    if os.path.exists(out_path):
        with open(out_path) as fr:
            for line in fr:
                try: done.add(json.loads(line)["task"])
                except Exception: pass
        print(f"resuming: {len(done)} problems already in {out_path}")

    new_pairs, attempted, solved = [], 0, 0
    with open(out_path, "a") as fout:
        for p in todo:
            if p["name"] in done:
                continue
            attempted += 1
            seen, kept = set(), 0
            for body in translate_candidates(p, n=n):
                if body in seen:
                    continue
                seen.add(body)
                if evaluate_one(p["prompt"], body, p["tests"]) == "pass":
                    rec = {"task": p["name"], "python": p["python"],
                           "rust_prompt": p["prompt"],
                           "rust_solution": p["prompt"] + body + "}"}
                    new_pairs.append(rec)
                    fout.write(json.dumps(rec) + "\n"); fout.flush()
                    kept += 1
            solved += (kept > 0)
            if attempted % 10 == 0:
                print(f"  {attempted} new problems | {len(new_pairs)} new pairs kept | {solved} solved this run")
    print(f"\nDONE this run: {len(new_pairs)} new pairs from {solved}/{attempted} problems -> {out_path}")
    return new_pairs

# Smoke-test first (cheap):  build_pairs(limit=20, n=8)
# Full run: 354 problems x 8 samples, keep ALL passing. Incremental + resumable.
# Does NOT touch the original pairs.jsonl. ~30-60+ min on a T4; if it disconnects, re-run this cell.
pairs = build_pairs(limit=354, n=4, out_path="pairs_v2.jsonl")


  10 new problems | 18 new pairs kept | 7 solved this run
  20 new problems | 37 new pairs kept | 16 solved this run
  30 new problems | 58 new pairs kept | 24 solved this run
  40 new problems | 83 new pairs kept | 32 solved this run
  50 new problems | 100 new pairs kept | 40 solved this run
  60 new problems | 116 new pairs kept | 48 solved this run
  70 new problems | 134 new pairs kept | 56 solved this run
  80 new problems | 160 new pairs kept | 65 solved this run
  90 new problems | 180 new pairs kept | 72 solved this run
  100 new problems | 206 new pairs kept | 81 solved this run
  110 new problems | 222 new pairs kept | 87 solved this run
  120 new problems | 246 new pairs kept | 96 solved this run
  130 new problems | 265 new pairs kept | 103 solved this run
  140 new problems | 279 new pairs kept | 108 solved this run
  150 new problems | 293 new pairs kept | 115 solved this run
  160 new problems | 313 new pairs kept | 123 solved this run
  170 new problems | 331 new pairs

## 6. Save the validated dataset
The engine above already streams each kept pair into **`pairs_v2.jsonl`** as it runs (crash-safe),
so this file is complete the moment the run finishes. We keep it **separate from the original
`pairs.jsonl`** (the 194-pair set behind our current result) so that dataset stays reproducible.
Point Step 3 at `pairs_v2.jsonl` for the next fine-tune.

In [7]:
import json
# pairs_v2.jsonl was written incrementally during the run above.
with open("pairs_v2.jsonl") as f:
    all_pairs = [json.loads(l) for l in f]
print("total validated pairs in pairs_v2.jsonl:", len(all_pairs))

if all_pairs:
    print("\n=== Example validated (Python -> Rust) pair ===")
    print("PYTHON:\n", all_pairs[0]["python"])
    print("\nRUST (compiles + passes tests):\n", all_pairs[0]["rust_solution"])

# Save the NEW file to Drive (kept separate from the original pairs.jsonl).
# from google.colab import drive; drive.mount("/content/drive")
# import shutil; shutil.copy("pairs_v2.jsonl", "/content/drive/MyDrive/CodeGen_Group45/pairs_v2.jsonl")
# print("copied pairs_v2.jsonl to Drive")


total validated pairs in pairs_v2.jsonl: 628

=== Example validated (Python -> Rust) pair ===
PYTHON:
 import math
def is_not_prime(n):
    result = False
    for i in range(2,int(math.sqrt(n)) + 1):
        if n % i == 0:
            result = True
    return result

RUST (compiles + passes tests):
 /// Write a rsthon function to identify non-prime numbers.
fn is_not_prime(n: isize) -> bool {
    let mut result = false;
    let sqrt_n: isize = (n as f64).sqrt() as isize + 1;
    for i in 2..sqrt_n {
        if n % i == 0 {
            result = true;
            break;
        }
    }
    result
}


## What we built (and what's next)
- A **data engine** at full scale: 354 MBPP problems x 8 sampled Rust attempts each, **validated by
  our harness**, keeping every passing solution -> **`pairs_v2.jsonl`**.
- The run is **incremental and resumable**: a Colab disconnect loses nothing — just re-run the engine cell.

**Next — Step 3:** set `pairs.jsonl` -> `pairs_v2.jsonl` in the Step 3 notebook, re-run the LoRA
fine-tune on the larger set, and re-measure on **HumanEval-Rust** (our single eval set) — the goal is
to move pass rate off the 1.3% floor.

**Mentor notes:** (1) we train on MBPP and evaluate on HumanEval — disjoint, no leakage; (2) the harness
doubles as the data-quality filter, so even an imperfect translator yields execution-verified data
(the MultiPL-T recipe).